In [ ]:
# TODO: move to import
# utility function to plot an isomorphism between two graphs
def plot_isomorphism(G, H, f, m, G_pos=None, H_pos=None):
    if G_pos == None:
        G_pos = nx.circular_layout(G)
    if H_pos == None:
        H_pos = nx.circular_layout(H)

    x_max = max(list(G_pos.values()), key=lambda v: v[0])[0]
    x_min = min(list(H_pos.values()), key=lambda v: v[0])[0]

    right_offset = (x_max - x_min) + 1
    for h, (x, y) in H_pos.items():
        H_pos[h] = (x+right_offset, y)

    nx.draw(G, with_labels=True, pos=G_pos)
    nx.draw(H, with_labels=True, pos=H_pos)

    arrows = []
    for g in G.nodes():
        h = m.eval(f(g)).as_long()
        h_pos = H_pos[h]
        g_pos = G_pos[g]
        arrow = patches.FancyArrowPatch(g_pos, h_pos, arrowstyle="-|>",
                                        linestyle=(0, (6, 4)),
                                        color="blue",
                                        connectionstyle="arc3,rad=-0.25",
                                        shrinkA=10, shrinkB=10,
                                        mutation_scale=20)
        arrows.append(arrow)

    for arrow in arrows:
        plt.gca().add_patch(arrow)

    plt.show()

# The Graph Isomorphism Problem

For two graphs $G$ and $H$, an isomorphism is a function $f$ that maps every node in graph $G$ to a node in graph $H$ such that each edge in $G$, corresponds to an edge in $H$.

More formally:

An isomorphism $ f: V(G) \rightarrow V(H) $ is a bijection such that
$$ (u, v) \in E(G) \iff (f(u), f(v)) \in E(H). $$
We call this function "structure-preserving" or "edge-preserving."
If two graphs have an isomorphism between them, they are said to be isomorphic.

A bijective function $f: X \rightarrow Y$ has the following properties:

$$ \forall x \in X \left( f^{-1}(f(x)) = x \right) $$
$$ \forall y \in Y \left( f(f^{-1}(y)) = y \right) $$

where $f^{-1}$ is the inverse of $f$.

For example, these two graphs are isomorphic:

In [ ]:
G = nx.Graph()
G.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 6),
         (2, 3), (2, 5), (3, 4),
         (3, 8), (4, 7), (5, 6),
         (5, 8), (6, 7), (7, 8)]
G.add_edges_from(edges)
G_pos = {1: (0, 3), 2: (1, 3), 3: (0, 2), 4: (1, 2), 5: (0, 1), 6: (1, 1), 7: (0, 0), 8: (1, 0)}

H = nx.Graph()
H.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 5),
         (2, 3), (2, 6), (3, 4),
         (3, 7), (4, 8), (5, 6),
         (5, 8), (6, 7), (7, 8)]
H.add_edges_from(edges)
H_pos = {1: (0, 3), 2: (3, 3), 3: (3, 0), 4: (0, 0), 5: (1, 2), 6: (2, 2), 7: (2, 1), 8: (1, 1)}

plt.subplot(1, 2, 1)
nx.draw(G, with_labels=True, pos=G_pos)

plt.subplot(1, 2, 2)
nx.draw(H, with_labels=True, pos=H_pos)

But these two are not:

In [ ]:
G = nx.complete_graph(5)
H = nx.complete_graph(5)
H.remove_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])

plt.subplot(1, 2, 1)
nx.draw(G, with_labels=True)

plt.subplot(1, 2, 2)
nx.draw_circular(H, with_labels=True)

# Making a Graph Isomorphism Solver

We will use Z3 to find a isomorphism between two graphs $G$ and $H$. We can use Z3's `Function` datatype in order to find a function from nodes in $G$ to nodes in $H$.

However, we will need to restrict the input and output of our function to be limited to nodes in $G$ and nodes in $H$ respectively.
We call this property "closure."

This can be expressed as:

$$ \forall g \in G,  \exists h \in H \left( f(g) = h \right) $$
$$ \forall h \in H, \exists g \in G \left( f^{-1}(h) = g \right) $$

We will use `Or` statements to encode the existential ($\exists$) constraints and for loops to encode the universal ($\forall$) constraints.

In [ ]:
# finds an isomorphism between two undirected graphs
def find_isomorphism(G, H, solver):

    # assumes graph labels are integers
    f = Function('f', IntSort(), IntSort())     # isomorphism f
    f_1 = Function('f_1', IntSort(), IntSort()) # inverse of f (f^(-1))

    for g in G.nodes():
        solver.add(f_1(f(g)) == g)                      # bijection constraint
        solver.add(Or([f(g) == h for h in H.nodes()]))  # closure constraint

    for h in H.nodes():
        solver.add(f(f_1(h)) == h)                          # bijection constraint
        solver.add(Or([f_1(h) == g for g in G.nodes()]))    # closure constraint

    # edge-preserving constraint
    for u_g, v_g in G.edges():
        solver.add(Or([
            Or(And(f(u_g) == u_h, f(v_g) == v_h),   # need to check both directions,
               And(f(u_g) == v_h, f(v_g) == u_h))   # assuming undirected graph
            for u_h, v_h in H.edges()
        ]))

    return f

We can test our solver with two graphs that we know are isomorphic:

In [ ]:
s = Solver()
G = nx.complete_graph(3)
H = nx.complete_graph(3)

f = find_isomorphism(G, H, s)

print(s.check()) # should print sat
m = s.model()
print(m)
plot_isomorphism(G, H, f, m)

And two graphs that we know are not isomorphic:

In [ ]:
s = Solver()
G = nx.complete_graph(5)
H = nx.complete_graph(5)
H.remove_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])

f = find_isomorphism(G, H, s)

print(s.check()) # should print unsat

Now we can find the isomorphism between the two graphs from before:

In [ ]:
s = Solver()
G = nx.Graph()
G.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 6),
         (2, 3), (2, 5), (3, 4),
         (3, 8), (4, 7), (5, 6),
         (5, 8), (6, 7), (7, 8)]
G.add_edges_from(edges)
G_pos = {1: (0, 3), 2: (1, 3), 3: (0, 2), 4: (1, 2), 5: (0, 1), 6: (1, 1), 7: (0, 0), 8: (1, 0)}

H = nx.Graph()
H.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 5),
         (2, 3), (2, 6), (3, 4),
         (3, 7), (4, 8), (5, 6),
         (5, 8), (6, 7), (7, 8)]
H.add_edges_from(edges)
H_pos = {1: (0, 3), 2: (3, 3), 3: (3, 0), 4: (0, 0), 5: (1, 2), 6: (2, 2), 7: (2, 1), 8: (1, 1)}

f = find_isomorphism(G, H, s)

print(s.check()) # should print sat
m = s.model()
print(m)
plot_isomorphism(G, H, f, m, G_pos=G_pos, H_pos=H_pos)